In [1]:
from src.module.database import oracle_import, oracle_export, oracle_execute
import json
import os
import sys
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text

In [19]:
oracle_execute("grant select on uni_ml_report.credit_collection_score to uni_ml_scoring")

2026-07-03 16:34:54.427 | INFO     | __main__:<module>:1 - Function oracle_execute executed in: 139 ms


In [2]:
p_date = str(datetime.today().date() -timedelta(days = 3)).replace("-", "")
p_date

'20260703'

In [3]:
consumer_activities = oracle_import(f"select * from toki.marketplace_consumer_activities where p_date = '{p_date}'")

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-07-06 15:15:41.563 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 5 sec 273 ms


In [4]:
consumer_activities["ACTIVITYNAME"].unique()

<StringArray>
['cart-events', 'limit-events', 'order-events', 'wishlist-events']
Length: 4, dtype: str

In [5]:
consumer_activities.groupby("ACTIVITYNAME").count()

,ID_,ACTIVITYDATA,CREATEDAT,UPDATEDAT,P_DATE
ACTIVITYNAME,,,,,
cart-events,101237,101237,101237,101237,101237
limit-events,3450,3450,3450,3450,3450
order-events,634,634,634,634,634
wishlist-events,90,90,90,90,90


In [6]:
consumer_activities.tail()

,ID_,ACTIVITYNAME,ACTIVITYDATA,CREATEDAT,UPDATEDAT,P_DATE
105406,6a47a4f2ce31add3c349798c,cart-events,"{'cartId': '689196be62785e5bae261f92', 'type':...",2026-07-03 20:02:58,2026-07-03 20:02:58,20260703
105407,6a47a4f34229463e57e52b9e,cart-events,"{'cartId': '6755a7c599e4f67868ac37ca', 'type':...",2026-07-03 20:02:59,2026-07-03 20:02:59,20260703
105408,6a47a4f3420fe633e02fc6c2,cart-events,"{'cartId': '615478fae0ddd83198ee21e9', 'type':...",2026-07-03 20:02:59,2026-07-03 20:02:59,20260703
105409,6a47a4f3805151b025dc269b,cart-events,"{'cartId': '62d37188aefa21e0acd79899', 'type':...",2026-07-03 20:02:59,2026-07-03 20:02:59,20260703
105410,6a47a4f3420fe633e02fc6c4,cart-events,"{'cartId': '626b6b7e892fd2df7aa8cb60', 'type':...",2026-07-03 20:02:59,2026-07-03 20:02:59,20260703


In [7]:
consumer_activities["ACTIVITYDATA"]  = consumer_activities["ACTIVITYDATA"].apply(lambda x: str(x).replace("'", "\"").replace("None", "null").replace("True", "true").replace("False", "false") if isinstance(x, str) else x)

In [8]:
json.loads(str(consumer_activities["ACTIVITYDATA"].values[0]))

{'cartId': '692eedb0da1ffae38df9cef0',
 'type': 'PRODUCT_MODIFIED',
 'item': {'productId': '692e313a49a5eecb319a3f55',
  'qty': 1,
  'available': True,
  '_id': '6a14878bce94f86945776bbb'},
 'cart': {'_id': '692eedc134e745504f6ef645',
  'accountId': '692eedb0da1ffae38df9cef0',
  'items': [{'productId': '699e6966fd48a0ea61c5e446',
    'qty': 1,
    'available': True,
    '_id': '6a09a4a9d4b9cb0c4e91c6b1'},
   {'productId': '692e313c49a5eecb319a3f58',
    'qty': 1,
    'available': True,
    '_id': '6a148775496155f535a34a16'},
   {'productId': '692e313a49a5eecb319a3f55',
    'qty': 1,
    'available': True,
    '_id': '6a14878bce94f86945776bbb'}],
  'createdAt': '2025-12-02T13:46:41.147Z',
  'updatedAt': '2026-07-02T20:04:16.829Z'}}

In [9]:
def safe_json_loads(x):
    if not isinstance(x, str):
        return x
    try:
        return json.loads(x)
    except (json.JSONDecodeError, ValueError):
        return None

consumer_activities["ACTIVITYDATA"] = consumer_activities["ACTIVITYDATA"].apply(safe_json_loads)

In [10]:
consumer_activities["ACTIVITYDATA"].values[0] #

{'cartId': '692eedb0da1ffae38df9cef0',
 'type': 'PRODUCT_MODIFIED',
 'item': {'productId': '692e313a49a5eecb319a3f55',
  'qty': 1,
  'available': True,
  '_id': '6a14878bce94f86945776bbb'},
 'cart': {'_id': '692eedc134e745504f6ef645',
  'accountId': '692eedb0da1ffae38df9cef0',
  'items': [{'productId': '699e6966fd48a0ea61c5e446',
    'qty': 1,
    'available': True,
    '_id': '6a09a4a9d4b9cb0c4e91c6b1'},
   {'productId': '692e313c49a5eecb319a3f58',
    'qty': 1,
    'available': True,
    '_id': '6a148775496155f535a34a16'},
   {'productId': '692e313a49a5eecb319a3f55',
    'qty': 1,
    'available': True,
    '_id': '6a14878bce94f86945776bbb'}],
  'createdAt': '2025-12-02T13:46:41.147Z',
  'updatedAt': '2026-07-02T20:04:16.829Z'}}

In [11]:
consumer_events = oracle_import(f"select * from toki.marketplace_consumer_EVENTS where p_date ={p_date} ")

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-07-06 15:17:22.894 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 1 min 33 sec 


In [12]:
p_date

'20260703'

In [13]:
consumer_events.shape

(14141, 11)

In [14]:
consumer_events.groupby("EVENTNAME").count()

,ID_,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
EVENTNAME,,,,,,,,,,
product_click,6261,6261,6261,6202,6261,6261,6261,6261,6261,6261
taxon_click,7880,7880,7880,7805,7880,7880,7880,7880,7880,7880


In [15]:
import config as cfg
import pandas as pd
import numpy as np
from src.database import pgsql_import

In [16]:
full_catalog_data = pgsql_import("select * from marketplace_catalog_data_extended_version3_staging")

In [17]:
full_catalog_data.shape

(4126, 52)

In [18]:
full_catalog_data.head()

,carried_located_in,main_category,sub_category,product_category,exact_product_category,manufacturer,generic_name,actual_product,size,power_consumption,...,color,image,productstate,createdat,updatedat,productmeta,saleprice,image_urls,details_translation,group_id
0,Living Room,Electronics,Televisions,Mini LED QLED 4K TVs,Sony 85XR50 85-inch Mini LED QLED 4K HDR Googl...,SONY,Smart Television,Mini LED QLED 4K HDR Google 85 inch tv /SONY-K...,85 inches,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9769a9ad-9471-5347-abf4-c920d69a0e3e
1,Living Room,Electronics,Televisions,4K UHD TVs,Full Array LED 4K HDR Smart TV,Panasonic,Smart Television,Panasonic TH-75NX900,75 inch,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,eae2fd2c-1c51-56f4-a7ec-7561dbfd11e9
2,Living Room,Electronics,Televisions,4K UHD TVs,Full Array LED TVs,Panasonic,Smart Television,Panasonic TH-55NX900 Full Array Led 55 inch sm...,55 inches,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,278b1e1a-844e-5fe0-ba34-76ea6f448aef
3,Living Room,Electronics,Televisions,Mini LED / QLED 4K TVs,Mini LED QLED 4K HDR Google TV (Sony XR90 series),SONY,Smart Television,Mini LED QLED 4K HDR Google 85 inch TV /SONY-K...,85 inches,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1840c5f5-1358-5611-8978-af4464e30d73
4,Studio,Electronics,Computer Audio,Microphones,Condenser Microphone,Sennheiser,Microphone,Sennheiser MK4 Condenser Microphone,,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,291e484e-b9e9-5b57-8c03-7e6b549927a9


## Snapshot Analysis Direction

The goal of this notebook section is to turn yesterday's activities/events plus the full catalog into one canonical interaction dataset. That interaction table becomes the contract for a realtime consumer pipeline later: every incoming event should map to `account_id`, `product_id` or `taxonomy/category`, `event_type`, `event_ts`, `weight`, and optional context.


In [19]:
# Quick schema and volume checks
print("consumer_activities", consumer_activities.shape)
print("consumer_events", consumer_events.shape)
print("full_catalog_data", full_catalog_data.shape)

print("\nActivity columns")
display(pd.DataFrame({"column": consumer_activities.columns, "dtype": consumer_activities.dtypes.astype(str).values}))

print("\nEvent columns")
display(pd.DataFrame({"column": consumer_events.columns, "dtype": consumer_events.dtypes.astype(str).values}))

print("\nCatalog columns")
display(pd.DataFrame({"column": full_catalog_data.columns, "dtype": full_catalog_data.dtypes.astype(str).values}))


consumer_activities (105411, 6)
consumer_events (14141, 11)
full_catalog_data (4126, 52)

Activity columns


,column,dtype
0,ID_,str
1,ACTIVITYNAME,str
2,ACTIVITYDATA,object
3,CREATEDAT,datetime64[us]
4,UPDATEDAT,datetime64[us]
5,P_DATE,str



Event columns


,column,dtype
0,ID_,str
1,EVENTNAME,str
2,EVENTVALUE,str
3,ACCOUNTID,str
4,SESSIONID,str
5,TIMESTAMP_,str
6,USERAGENT,str
7,URL_,str
8,CREATEDAT,datetime64[us]
9,UPDATEDAT,datetime64[us]



Catalog columns


,column,dtype
0,carried_located_in,str
1,main_category,str
2,sub_category,str
3,product_category,str
4,exact_product_category,str
5,manufacturer,str
6,generic_name,str
7,actual_product,str
8,size,str
9,power_consumption,str


In [20]:
# Inspect nested ACTIVITYDATA keys per activity type. This tells us which fields can feed the stream contract.
def flatten_keys(value, prefix=""):
    keys = set()
    if isinstance(value, dict):
        for key, nested in value.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            keys.add(path)
            keys.update(flatten_keys(nested, path))
    elif isinstance(value, list):
        for item in value[:3]:
            keys.update(flatten_keys(item, f"{prefix}[]" if prefix else "[]"))
    return keys

activity_key_profile = (
    consumer_activities
    .assign(_keys=consumer_activities["ACTIVITYDATA"].apply(flatten_keys))
    .groupby("ACTIVITYNAME")["_keys"]
    .agg(lambda rows: sorted(set().union(*rows)))
)

for activity_name, keys in activity_key_profile.items():
    print(f"\n{activity_name}: {len(keys)} keys")
    print(keys[:80])



cart-events: 23 keys
['cart', 'cart._id', 'cart.accountId', 'cart.createdAt', 'cart.items', 'cart.items[]._id', 'cart.items[].available', 'cart.items[].productId', 'cart.items[].qty', 'cart.updatedAt', 'cartId', 'item', 'item._id', 'item.accountId', 'item.available', 'item.items', 'item.items[].available', 'item.items[].productId', 'item.items[].qty', 'item.items[].storeId', 'item.productId', 'item.qty', 'type']

limit-events: 5 keys
['accountId', 'result', 'result.expireDate', 'result.limit', 'result.status']

order-events: 13 keys
['data', 'data.accountId', 'data.deliveryMethod', 'data.items', 'data.items[].name', 'data.items[].productId', 'data.items[].qty', 'data.items[].transNumber', 'data.items[].unitPrice', 'data.orderId', 'data.orderNo', 'data.vendorOrderId', 'type']

wishlist-events: 3 keys
['accountId', 'item', 'type']


In [21]:
# Normalize activity rows into account/product interactions.
def nested_get(value, path, default=None):
    current = value
    for part in path.split("."):
        if not isinstance(current, dict) or part not in current:
            return default
        current = current[part]
    return current

def normalize_activity_row(row):
    data = row.get("ACTIVITYDATA") if isinstance(row.get("ACTIVITYDATA"), dict) else {}
    activity_name = row.get("ACTIVITYNAME")
    event_type = data.get("type") or activity_name
    account_id = data.get("accountId") or nested_get(data, "cart.accountId")
    product_id = nested_get(data, "item.productId")

    if product_id is None and isinstance(nested_get(data, "cart.items"), list) and data["cart"]["items"]:
        product_id = data["cart"]["items"][0].get("productId")

    return {
        "source": "consumer_activities",
        "source_id": row.get("ID_"),
        "account_id": account_id,
        "session_id": None,
        "event_name": activity_name,
        "event_type": event_type,
        "product_id": product_id,
        "event_value": None,
        "event_ts": row.get("CREATEDAT"),
        "p_date": row.get("P_DATE"),
    }

activity_interactions = pd.DataFrame(
    normalize_activity_row(row) for _, row in consumer_activities.iterrows()
)

activity_interactions.head()


,source,source_id,account_id,session_id,event_name,event_type,product_id,event_value,event_ts,p_date
0,consumer_activities,6a46c440ce31add3c348fc5c,692eedb0da1ffae38df9cef0,None,cart-events,PRODUCT_MODIFIED,692e313a49a5eecb319a3f55,None,2026-07-03 04:04:16,20260703
1,consumer_activities,6a46c4408c8c31c8475ede73,692ee1e726c36fdc187d48bc,None,cart-events,PRODUCT_MODIFIED,6916c714c27e96867b19420d,None,2026-07-03 04:04:16,20260703
2,consumer_activities,6a46c440ce31add3c348fc5e,692eedb0da1ffae38df9cef0,None,cart-events,PRODUCT_MODIFIED,692e313c49a5eecb319a3f58,None,2026-07-03 04:04:16,20260703
3,consumer_activities,6a46c4408c8c31c8475ede71,61c581aac4e8e78c180df5f1,None,cart-events,PRODUCT_MODIFIED,6916c70ac27e96867b1941ff,None,2026-07-03 04:04:16,20260703
4,consumer_activities,6a46c440420fe633e02f4a12,638dac7d56f427af73683bdd,None,cart-events,PRODUCT_MODIFIED,6911b05be91ec7bfb3daf519,None,2026-07-03 04:04:16,20260703


In [22]:
# Normalize click events. EVENTVALUE may be product id, taxon/category id, slug, or JSON depending on source.
def safe_event_value(value):
    if isinstance(value, dict):
        return value
    if not isinstance(value, str):
        return value
    try:
        return json.loads(value)
    except (json.JSONDecodeError, ValueError):
        return value

def normalize_event_row(row):
    value = safe_event_value(row.get("EVENTVALUE"))
    product_id = None
    taxon_id = None

    if isinstance(value, dict):
        product_id = value.get("productId") or value.get("product_id") or value.get("id")
        taxon_id = value.get("taxonId") or value.get("taxon_id") or value.get("categoryId")
    elif row.get("EVENTNAME") == "product_click":
        product_id = value
    elif row.get("EVENTNAME") == "taxon_click":
        taxon_id = value

    return {
        "source": "consumer_events",
        "source_id": row.get("ID_"),
        "account_id": row.get("ACCOUNTID"),
        "session_id": row.get("SESSIONID"),
        "event_name": row.get("EVENTNAME"),
        "event_type": row.get("EVENTNAME"),
        "product_id": product_id,
        "taxon_id": taxon_id,
        "event_value": value,
        "event_ts": row.get("TIMESTAMP_") or row.get("CREATEDAT"),
        "p_date": row.get("P_DATE"),
    }

event_interactions = pd.DataFrame(
    normalize_event_row(row) for _, row in consumer_events.iterrows()
)

event_interactions.head()


,source,source_id,account_id,session_id,event_name,event_type,product_id,taxon_id,event_value,event_ts,p_date
0,consumer_events,6a4695af420fe633e02f32fc,69b42f06907ef4770a9be935,gwNRxtmmO1qpi3tXyRw7Y5T2TNmE_Ks7,product_click,product_click,"{'productIds': ['6833c3440b23fc4fb994bf7c', '6...",NaN,"{'productIds': ['6833c3440b23fc4fb994bf7c', '6...",2026-07-02T16:45:35.466Z,20260703
1,consumer_events,6a4695b2420fe633e02f32fe,627de436db38f457813a5431,LqIAz_R31oXsj7ptxHB-9O8RPmvOJLty,product_click,product_click,"{'productIds': ['68d9dabb63d257398caccc78'], '...",NaN,"{'productIds': ['68d9dabb63d257398caccc78'], '...",2026-07-02T16:45:38.848Z,20260703
2,consumer_events,6a4695b48c8c31c8475ec7a1,60015c46c747de1e412c8f1c,NaN,taxon_click,taxon_click,NaN,{'taxon': {'label': 'Консоль'}},{'taxon': {'label': 'Консоль'}},2026-07-02T16:46:53.110Z,20260703
3,consumer_events,6a4695bbce31add3c348e5dc,60015c46c747de1e412c8f1c,NaN,taxon_click,taxon_click,NaN,{'taxon': {'label': 'Компьютер'}},{'taxon': {'label': 'Компьютер'}},2026-07-02T16:47:00.852Z,20260703
4,consumer_events,6a4695bf4aeec353170ec322,60015c46c747de1e412c8f1c,NaN,taxon_click,taxon_click,NaN,{'taxon': {'label': 'Сандал ширээ'}},{'taxon': {'label': 'Сандал ширээ'}},2026-07-02T16:47:04.813Z,20260703


In [23]:
# Canonical behavior table. Tune weights after validating business outcomes.
EVENT_WEIGHTS = {
    "product_click": 1.0,
    "taxon_click": 0.3,
    "cart-events": 3.0,
    "PRODUCT_ADDED": 4.0,
    "PRODUCT_MODIFIED": 2.0,
    "wishlist-events": 5.0,
    "order-events": 10.0,
    "limit-events": 1.5,
}

interactions = pd.concat([activity_interactions, event_interactions], ignore_index=True, sort=False)
interactions["event_ts"] = pd.to_datetime(interactions["event_ts"], errors="coerce", utc=True)
interactions["weight"] = interactions["event_type"].map(EVENT_WEIGHTS).fillna(
    interactions["event_name"].map(EVENT_WEIGHTS)
).fillna(1.0)
interactions["has_product_id"] = interactions["product_id"].notna() & (interactions["product_id"].astype(str).str.len() > 0)

summary = pd.DataFrame({
    "metric": [
        "rows",
        "unique_accounts",
        "product_level_rows",
        "rows_missing_account",
        "rows_missing_event_ts",
    ],
    "value": [
        len(interactions),
        interactions["account_id"].nunique(dropna=True),
        int(interactions["has_product_id"].sum()),
        int(interactions["account_id"].isna().sum()),
        int(interactions["event_ts"].isna().sum()),
    ]
})

display(summary)
display(interactions.groupby(["source", "event_name", "event_type"], dropna=False).agg(
    rows=("source_id", "count"),
    accounts=("account_id", "nunique"),
    product_rows=("has_product_id", "sum"),
    avg_weight=("weight", "mean"),
).sort_values("rows", ascending=False).head(30))


,metric,value
0,rows,119552
1,unique_accounts,26665
2,product_level_rows,107232
3,rows_missing_account,656
4,rows_missing_event_ts,0


rows  accounts  \
source              event_name      event_type                          
consumer_activities cart-events     PRODUCT_MODIFIED  99647     23083   
consumer_events     taxon_click     taxon_click        7880      2404   
                    product_click   product_click      6261      2161   
consumer_activities limit-events    limit-events       3450      1731   
                    cart-events     ITEM_ADDED          969       686   
                                    PRODUCT_ORDERED     290       221   
                                    ITEM_REMOVED        278       178   
                    order-events    PRODUCT_ORDERED     275         0   
                                    ORDER_EXPIRED       226         0   
                    wishlist-events ITEM_ADDED           71        58   
                    order-events    ORDER_COMPLETED      61         0   
                                    ORDER_ACTIVATED      60         0   
                    cart-events     PRODUCT_REMOVED      32        31   
                                    cart-events          21         0   
                    wishlist-events ITEM_REMOVED         19        17   
                    order-events    order-events         12         0   

                                                      product_rows  avg_weight  
source              event_name      event_type                                  
consumer_activities cart-events     PRODUCT_MODIFIED         99646         2.0  
consumer_events     taxon_click     taxon_click                  0         0.3  
                    product_click   product_click             6261         1.0  
consumer_activities limit-events    limit-events                 0         1.5  
                    cart-events     ITEM_ADDED                 969         3.0  
                                    PRODUCT_ORDERED             46         3.0  
                                    ITEM_REMOVED               278         3.0  
                    order-events    PRODUCT_ORDERED              0        10.0  
                                    ORDER_EXPIRED                0        10.0  
                    wishlist-events ITEM_ADDED                   0         5.0  
                    order-events    ORDER_COMPLETED              0        10.0  
                                    ORDER_ACTIVATED              0        10.0  
                    cart-events     PRODUCT_REMOVED             32         3.0  
                                    cart-events                  0         3.0  
                    wishlist-events ITEM_REMOVED                 0         5.0  
                    order-events    order-events                 0        10.0

In [24]:
# Catalog readiness check. Select the strongest available product id column before joining.
candidate_product_columns = [
    col for col in full_catalog_data.columns
    if str(col).lower() in {"product_id", "productid", "id", "_id", "group_id"}
    or "product" in str(col).lower()
]

print(candidate_product_columns)

catalog_profile = full_catalog_data.agg(["count", "nunique"]).T.reset_index()
catalog_profile.columns = ["column", "non_null_rows", "unique_values"]
display(catalog_profile.sort_values(["non_null_rows", "unique_values"], ascending=False).head(30))


['product_category', 'exact_product_category', 'actual_product', 'product_id', 'productstate', 'productmeta', 'group_id']


,column,non_null_rows,unique_values
17,index,4126,4126
18,product_id,4126,4126
24,keywords,4126,4122
22,details,4126,4121
11,specifications,4126,4117
21,main_option,4126,4045
12,sku,4126,3669
7,actual_product,4126,3664
23,url_link,4126,3620
4,exact_product_category,4126,3094


In [25]:
# User preference profile from the normalized interactions.
# After confirming the catalog join key, enrich this with category/manufacturer/price attributes.
user_behavior_profile = (
    interactions[interactions["account_id"].notna()]
    .sort_values("event_ts")
    .groupby("account_id")
    .agg(
        total_events=("source_id", "count"),
        product_events=("has_product_id", "sum"),
        score=("weight", "sum"),
        first_seen=("event_ts", "min"),
        last_seen=("event_ts", "max"),
        distinct_products=("product_id", "nunique"),
        distinct_sessions=("session_id", "nunique"),
    )
    .sort_values("score", ascending=False)
)

user_behavior_profile.head(20)


,total_events,product_events,score,first_seen,last_seen,distinct_products,distinct_sessions
account_id,,,,,,,
60a5f829c0f4e6e102e118b4,85,85,170.0,2026-07-03 04:02:37+00:00,2026-07-03 20:02:42+00:00,19,0
67988957605b2e288061f34a,104,48,109.9,2026-07-03 05:10:10.518000+00:00,2026-07-03 17:26:48+00:00,16,1
6743128a122aa970153df400,74,54,102.4,2026-07-03 04:34:21.056000+00:00,2026-07-03 15:25:25+00:00,26,1
61d964e601ba542dfc55b925,45,44,97.3,2026-07-03 04:05:02+00:00,2026-07-03 15:25:52+00:00,12,1
692998ef26c36fdc1861e0a1,41,41,82.0,2026-07-03 04:02:58+00:00,2026-07-03 20:03:06+00:00,13,0
63b2da530fc2230868ef7ef6,68,36,70.3,2026-07-02 23:39:13.623000+00:00,2026-07-03 12:41:16.281000+00:00,19,1
5fa0ea6e2280fe45d1195ab3,68,51,64.4,2026-07-03 14:09:19.996000+00:00,2026-07-03 23:37:08+00:00,18,1
6a446d286b1d21d482018609,50,31,62.7,2026-07-03 03:24:19.273000+00:00,2026-07-03 14:06:18+00:00,11,1
5ff9b2f9fc269d212aedcb7a,73,54,62.1,2026-07-03 08:12:17.328000+00:00,2026-07-03 15:47:29.825000+00:00,31,1


In [26]:
user_behavior_profile.reset_index(inplace = True)

In [27]:
user_behavior_profile[user_behavior_profile['account_id'] == '67a2437aee01f1d0a24c225c']

,account_id,total_events,product_events,score,first_seen,last_seen,distinct_products,distinct_sessions


In [28]:
user_behavior_profile.shape

(26665, 8)

## Realtime Pipeline Strategy

1. Capture every marketplace behavior as an immutable event: product click, taxon click, add to cart, cart update, wishlist, order, limit/budget events, search, and recommendation impression/click.

2. Standardize the event contract before scoring. Required fields: `event_id`, `account_id`, `session_id`, `event_type`, `product_id`, `taxon_id`, `event_ts`, `source`, `context`, `p_date`. The notebook's `interactions` dataframe is the offline prototype of that contract.

3. Stream ingestion should write raw events first, then normalized events. Use raw storage for replay/debugging and normalized storage for recommendation features.

4. Maintain online features per user/session: recent product ids, recent categories/taxons, cart contents, wishlist products, order history, price affinity, brand affinity, negative signals, and last activity timestamp.

5. Generate candidates from multiple lanes: same-category products, similar products from catalog attributes, cart complements, wishlist substitutes, popular products within clicked taxons, trending products, and cold-start/editorial fallback.

6. Rank candidates with a business-aware score: user intent weight, recency decay, catalog similarity, stock/availability, price fit, diversity, margin/promotion boost if allowed, and suppression rules for purchased/unavailable/repeated impressions.

7. Post recommendations through a dedicated recommendation API or message topic. The scorer should emit `account_id`, `recommendation_id`, ranked products, reason codes, model_version, generated_at, and expiry.

8. Log impressions, clicks, add-to-cart, wishlist, and orders from recommendation surfaces. Without these feedback events, realtime personalization cannot improve safely.


## First Build Milestones

- Milestone 1: finish offline validation in this notebook by confirming product/catalog join keys and measuring coverage.
- Milestone 2: create a daily batch recommender using the same `interactions` contract and write results to a recommendation table.
- Milestone 3: add streaming normalization for new events and update online user/session features.
- Milestone 4: expose recommendations through API/topic and track recommendation impressions/clicks.
- Milestone 5: introduce ranking experiments and A/B metrics: CTR, add-to-cart rate, order conversion, revenue per session, freshness, and latency.
